# Corrective RAG：本地資料答不了怎麼辦？

先問菜單，再問下雨。注意兩種問題都會先試本地索引；只有文件無關的問題，才改寫並查外部來源。網路分支預設使用固定資料，不是即時天氣。

這是課程新寫的可重現實驗；資料為虛構餐廳示例。預設使用真正的 LangGraph、稀疏向量索引和可檢查的固定策略，不使用模型或真實搜尋服務。第一格需網路安裝套件，之後執行不需要金鑰。

## 1. 準備環境
Colab：選「執行階段 → 全部執行」。本機：依 README 建立 Python 3.12 環境。

In [ ]:
# Colab 第一次執行會安裝課程固定版本；本機已按 README 安裝時不連網。
from importlib.metadata import version, PackageNotFoundError
import sys, subprocess
try:
    installed = version("langgraph")
except PackageNotFoundError:
    installed = None
if installed != "1.2.20":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.20"])
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add
from math import sqrt

## 2. 檢查可以引用的資料
只有本地菜單、營業時間與一筆外部預報樣本。先找出你預期回答會引用的來源 ID。

In [ ]:
# 小型稀疏向量索引：維度是詞彙；向量餘弦相似度決定排序，不呼叫 embedding API。
LOCAL = [
    {"id":"menu", "kind":"menu", "tags":["菜單","素食","餐點"], "text":"素食菜單有絲瓜蒸餃。"},
    {"id":"hours", "kind":"hours", "tags":["營業","時間"], "text":"教學餐廳的營業時間為 11:00–20:00。"},
]
WEB = [{"id":"web-demo", "kind":"weather", "tags":["天氣","下雨"], "text":"課堂固定預報樣本：有雨。"}]
REFUSAL = "目前沒有足夠來源回答。"
VOCAB = sorted({tag for doc in LOCAL for tag in doc["tags"]})
def vector(query):
    return [float(tag in query) for tag in VOCAB]
def cosine(x, y):
    return sum(a*b for a,b in zip(x,y)) / (sqrt(sum(a*a for a in x)*sum(b*b for b in y)) or 1)
def search_local(query):
    scored = [(cosine(vector(query),vector(" ".join(d["tags"]))),d) for d in LOCAL]
    score, doc = max(scored, key=lambda pair: pair[0])
    return [doc] if score > 0 else []
def intent(question):
    for doc in LOCAL + WEB:
        if any(tag in question for tag in doc["tags"]):
            return doc["kind"]
    return "hours" if "打烊" in question else "unknown"
def rewritten(question):
    return {"hours":"營業時間","menu":"素食菜單","weather":"天氣樣本"}.get(intent(question), question)
def cite(doc):
    return doc["text"] + " [" + doc["id"] + "]"

class State(TypedDict, total=False):
    question: str
    query: str
    docs: list
    answer: str
    route: str
    relevant: bool
    verdict: str
    retries: int
    generations: int
    fault: str
    trace: Annotated[list[str], add]

def step(name, **values):
    return {"trace":[name], **values}
def retrieve(s):
    return step("Retrieve", docs=search_local(s["query"]))
def grade(s):
    ok = any(d["kind"] == intent(s["question"]) for d in s["docs"])
    return step("grade_documents", relevant=ok)
def transform(s):
    return step("transform_query", query=rewritten(s["question"]), retries=s.get("retries",0)+1)
def generate(s):
    doc = next((d for d in s["docs"] if d["kind"]==intent(s["question"])), None)
    return step("Generate", answer=cite(doc) if doc else REFUSAL, generations=s.get("generations",0)+1)
def abstain(s):
    return step("Abstain", answer=REFUSAL)
def web_fixture(s):
    # 固定外部結果，不會真的搜尋網路；未知主題一律回空。
    return step("Web Search (fixture)", docs=[d for d in WEB if any(t in s["query"] for t in d["tags"])])

def run(question, **kw):
    out=graph.invoke({"question":question,"query":question,**kw},{"recursion_limit":30})
    print(" → ".join(out["trace"]), "\n", out["answer"])
    return out

## 3. 把投影片的方塊和箭頭接成圖
每個節點回傳部分 state；每條條件邊決定下一步。追 trace 觀察實際走法。

In [ ]:
def grade_question(s):
    # 課程的 GradeQuestion：先辨認是否屬於這份餐廳資料，仍讓每題先試本地檢索。
    return step("GradeQuestion",route="local" if intent(s["question"]) in {"menu","hours"} else "other")
def build_crag(web_search=web_fixture, answer=generate):
    g=StateGraph(State)
    for name,fn in {"scope":grade_question,"retrieve":retrieve,"grade":grade,"rewrite":transform,
                    "search":web_search,"generate":answer}.items(): g.add_node(name,fn)
    g.add_edge(START,"scope"); g.add_edge("scope","retrieve"); g.add_edge("retrieve","grade")
    g.add_conditional_edges("grade",lambda s:"generate" if s["relevant"] else "rewrite",["generate","rewrite"])
    g.add_edge("rewrite","search"); g.add_edge("search","generate"); g.add_edge("generate",END)
    return g.compile()
graph=build_crag()

## 4. 執行驗收
斷言失敗表示實際分支或答案與規格不符。

In [ ]:
a=run("有什麼素食餐點？")
assert a["answer"]==cite(LOCAL[0]) and "Web Search (fixture)" not in a["trace"]
b=run("今天會下雨嗎？")
assert b["trace"]==["GradeQuestion","Retrieve","grade_documents","transform_query","Web Search (fixture)","Generate"]
assert b["answer"]==cite(WEB[0])
assert run("停車費多少？")["answer"]==REFUSAL

## 5. 自己動手
把 WEB 改為空清單，再問天氣。驗收：仍執行搜尋分支，但必須拒答，不能捏造天氣。

下一步：需要測真實模型時，依資料夾 README 的「Claude Agent SDK」執行獨立的 live graph；不會影響這本的無金鑰驗收。

## 選做：替換同一個搜尋節點，改用真正的 Tavily
前面的斷言刻意使用固定樣本，沒有上網。若要比較真實搜尋，先準備自己的 Tavily API key，將下一格 `RUN_TAVILY` 改成 `True` 再單獨執行。這會使用網路與服務額度。這格直接使用 Tavily Search API，對應圖中的 Web Search 節點；輸出網頁原文節錄和網址，不把搜尋摘要當成已驗證的預報。

In [ ]:
RUN_TAVILY = False
if RUN_TAVILY:
    import json, urllib.request
    from getpass import getpass

    def make_search(key):
        def tavily(s):
            query = s["question"] + " 氣象來源" if intent(s["question"])=="weather" else s["query"]
            req = urllib.request.Request("https://api.tavily.com/search",
                data=json.dumps({"query":query,"max_results":2,"search_depth":"basic"}).encode(),
                headers={"Content-Type":"application/json","Authorization":"Bearer "+key})
            with urllib.request.urlopen(req,timeout=20) as res:
                results=json.load(res).get("results",[])
            docs=[{"id":v["url"],"kind":"external","text":v["content"][:280]} for v in results
                  if v.get("url","").startswith("https://") and v.get("content")]
            return step("Web Search (Tavily)",docs=docs)
        return tavily

    def answer_with_excerpt(s):
        if not s["docs"] or s["docs"][0]["kind"]!="external":
            return generate(s)
        doc=s["docs"][0]
        return step("Generate", answer="找到待查核的外部原文："+cite(doc))

    live = build_crag(make_search(getpass("Tavily key（不顯示）：")), answer_with_excerpt)
    question="台北今天的天氣？"
    out=live.invoke({"question":question,"query":question})
    print(" → ".join(out["trace"]), out["answer"], sep="\n")
else:
    print("已略過付費連線；預設驗收使用固定搜尋樣本。")